In [1]:
import os
import re
import glob
import numpy as np
from typing import List, Optional, Generator, Tuple, Iterable

_CHUNK_NOISY_PATTERN = re.compile(r'kappa_noisy_chunk_(\d+)\.npy')
_CHUNK_LABEL_PATTERN = re.compile(r'label_chunk_(\d+)\.npy')

def get_chunk_indices(chunk_dir: str) -> List[int]:
    """Discover chunk indices that have both noisy and label files."""
    noisy_files = glob.glob(os.path.join(chunk_dir, 'kappa_noisy_chunk_*.npy'))
    label_files = glob.glob(os.path.join(chunk_dir, 'label_chunk_*.npy'))

    noisy_idx = {int(_CHUNK_NOISY_PATTERN.search(os.path.basename(p)).group(1))
                 for p in noisy_files if _CHUNK_NOISY_PATTERN.search(os.path.basename(p))}
    label_idx = {int(_CHUNK_LABEL_PATTERN.search(os.path.basename(p)).group(1))
                 for p in label_files if _CHUNK_LABEL_PATTERN.search(os.path.basename(p))}
    common = sorted(list(noisy_idx & label_idx))
    return common

def _paths_for_index(chunk_dir: str, idx: int) -> Tuple[str, str]:
    noisy_path = os.path.join(chunk_dir, f'kappa_noisy_chunk_{idx}.npy')
    label_path = os.path.join(chunk_dir, f'label_chunk_{idx}.npy')
    return noisy_path, label_path

def load_chunks(chunk_dir: str,
                indices: Optional[Iterable[int]] = None,
                mmap_mode: Optional[str] = None,
                concat_axis: int = 0,
                verbose: bool = False) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load and concatenate multiple chunk files into two arrays (noisy, label).
    WARNING: concatenation returns a single array and may use a lot of memory.
    Use iter_chunks or batch_generator for memory-limited workflows.
    """
    if indices is None:
        indices = get_chunk_indices(chunk_dir)
    noisy_list = []
    label_list = []
    for idx in indices:
        noisy_path, label_path = _paths_for_index(chunk_dir, idx)
        if not (os.path.exists(noisy_path) and os.path.exists(label_path)):
            raise FileNotFoundError(f"Missing chunk files for index {idx}: {noisy_path}, {label_path}")
        noisy = np.load(noisy_path, mmap_mode=mmap_mode)
        label = np.load(label_path, mmap_mode=mmap_mode)
        if noisy.shape[0] != label.shape[0]:
            raise ValueError(f"Chunk {idx} sample count mismatch: noisy {noisy.shape[0]} vs label {label.shape[0]}")
        noisy_list.append(np.array(noisy))  # force read for consistent typing if mmap_mode is None
        label_list.append(np.array(label))
        if verbose:
            print(f"Loaded chunk {idx}: noisy {noisy.shape}, label {label.shape}")
    if not noisy_list:
        return np.array([]), np.array([])
    noisy_all = np.concatenate(noisy_list, axis=concat_axis)
    label_all = np.concatenate(label_list, axis=concat_axis)
    return noisy_all, label_all

def iter_chunks(chunk_dir: str,
                indices: Optional[Iterable[int]] = None,
                mmap_mode: Optional[str] = None) -> Generator[Tuple[np.ndarray, np.ndarray, int], None, None]:
    """
    Generator that yields (noisy_array, label_array, chunk_idx) for each chunk.
    If mmap_mode is set (e.g. 'r'), the yielded noisy/label may be numpy.memmap objects which support slicing.
    """
    if indices is None:
        indices = get_chunk_indices(chunk_dir)
    for idx in indices:
        noisy_path, label_path = _paths_for_index(chunk_dir, idx)
        if not (os.path.exists(noisy_path) and os.path.exists(label_path)):
            raise FileNotFoundError(f"Missing chunk files for index {idx}: {noisy_path}, {label_path}")
        noisy = np.load(noisy_path, mmap_mode=mmap_mode)
        label = np.load(label_path, mmap_mode=mmap_mode)
        if noisy.shape[0] != label.shape[0]:
            raise ValueError(f"Chunk {idx} sample count mismatch: noisy {noisy.shape[0]} vs label {label.shape[0]}")
        yield noisy, label, idx

def batch_generator(chunk_dir: str,
                    batch_size: int,
                    indices: Optional[Iterable[int]] = None,
                    shuffle: bool = True,
                    shuffle_mode: str = 'per_chunk',  # 'per_chunk' or 'global'
                    mmap_mode: Optional[str] = 'r',
                    seed: Optional[int] = None) -> Generator[Tuple[np.ndarray, np.ndarray], None, None]:
    """
    Yield minibatches (noisy_batch, label_batch).
    - per_chunk: iterate chunks (possibly shuffling samples inside each chunk).
    - global: build a global list of (chunk_idx, local_idx) then shuffle and yield batches (requires mmap_mode to avoid heavy memory use).
    mmap_mode is recommended (e.g. 'r') for streaming.
    """
    rng = np.random.default_rng(seed)
    if indices is None:
        indices = get_chunk_indices(chunk_dir)
    indices = list(indices)

    if shuffle_mode not in ('per_chunk', 'global'):
        raise ValueError("shuffle_mode must be 'per_chunk' or 'global'")

    if shuffle_mode == 'per_chunk':
        # Optionally shuffle chunk order
        chunk_order = indices.copy()
        if shuffle:
            rng.shuffle(chunk_order)
        for chunk_idx in chunk_order:
            noisy_path, label_path = _paths_for_index(chunk_dir, chunk_idx)
            noisy = np.load(noisy_path, mmap_mode=mmap_mode)
            label = np.load(label_path, mmap_mode=mmap_mode)
            n = noisy.shape[0]
            order = np.arange(n)
            if shuffle:
                rng.shuffle(order)
            # yield batches within this chunk
            for start in range(0, n, batch_size):
                batch_idx = order[start:start + batch_size]
                yield noisy[batch_idx], label[batch_idx]

    else:  # global shuffle across all chunks
        # Build mapping (chunk_idx, local_idx). This can be large but stores only pairs of ints.
        index_map = []
        lengths = {}
        for chunk_idx in indices:
            noisy_path, label_path = _paths_for_index(chunk_dir, chunk_idx)
            noisy = np.load(noisy_path, mmap_mode=mmap_mode)
            label = np.load(label_path, mmap_mode=mmap_mode)
            if noisy.shape[0] != label.shape[0]:
                raise ValueError(f"Chunk {chunk_idx} sample count mismatch: noisy {noisy.shape[0]} vs label {label.shape[0]}")
            n = noisy.shape[0]
            lengths[chunk_idx] = n
            index_map.extend((chunk_idx, i) for i in range(n))
        if shuffle:
            rng.shuffle(index_map)
        # open memmaps and keep in cache to avoid repeated disk opens
        memmaps = {}
        for start in range(0, len(index_map), batch_size):
            batch_pairs = index_map[start:start + batch_size]
            # group by chunk for efficient slicing
            groups = {}
            for cidx, lidx in batch_pairs:
                groups.setdefault(cidx, []).append(lidx)
            batch_noisy_parts = []
            batch_label_parts = []
            for cidx, local_idxs in groups.items():
                if cidx not in memmaps:
                    noisy_path, label_path = _paths_for_index(chunk_dir, cidx)
                    memmaps[cidx] = (np.load(noisy_path, mmap_mode=mmap_mode),
                                     np.load(label_path, mmap_mode=mmap_mode))
                noisy_mem, label_mem = memmaps[cidx]
                idx_arr = np.array(local_idxs, dtype=np.intp)
                batch_noisy_parts.append(noisy_mem[idx_arr])
                batch_label_parts.append(label_mem[idx_arr])
            # concatenate order must match batch_pairs order
            # Reconstruct batch in correct order:
            batch_noisy = []
            batch_label = []
            for cidx, lidx in batch_pairs:
                # find the position inside groups[cidx] to extract the corresponding row
                # faster approach: use dict[(cidx, lidx)] -> position, but for small batch sizes this is fine:
                # find index of lidx in groups[cidx] (but groups[cidx] may have duplicates; use pop approach)
                pass
            # Simpler: gather by iterating batch_pairs and indexing memmaps
            batch_noisy = np.stack([memmaps[cidx][0][lidx] for cidx, lidx in batch_pairs], axis=0)
            batch_label = np.stack([memmaps[cidx][1][lidx] for cidx, lidx in batch_pairs], axis=0)
            yield batch_noisy, batch_label

In [ ]:
import os
import numpy as np
from typing import Optional, Tuple, Generator, Dict, Any
from sklearn.model_selection import train_test_split  # For splitting; assumes scikit-learn is available
# Assuming PyTorch for model training; adjust if using another framework
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Assuming the previous functions (get_chunk_indices, iter_chunks, etc.) are available
# If not, include them or adapt accordingly.

def array_batch_generator(noisy_array: np.ndarray, label_array: np.ndarray, 
                          batch_size: int, shuffle: bool = False, 
                          seed: Optional[int] = None) -> Generator[Tuple[np.ndarray, np.ndarray], None, None]:
    """
    Simple batch generator for in-memory numpy arrays (e.g., after loading/splitting a chunk).
    Yields (noisy_batch, label_batch) as numpy arrays.
    """
    rng = np.random.default_rng(seed)
    n = noisy_array.shape[0]
    indices = np.arange(n)
    if shuffle:
        rng.shuffle(indices)
    
    for start in range(0, n, batch_size):
        batch_idx = indices[start:start + batch_size]
        yield noisy_array[batch_idx], label_array[batch_idx]

def incremental_train(model: nn.Module, 
                      optimizer: optim.Optimizer, 
                      criterion: nn.Module, 
                      chunk_dir: str,
                      split_ratio: float = 0.8,
                      batch_size: int = 128,
                      epochs_per_chunk: int = 1,
                      device: str = 'mps' if torch.mps.is_available() else 'cpu',
                      verbose: bool = False) -> Dict[str, Any]:
    """
    Incrementally train a PyTorch model across chunks without loading the entire dataset.
    
    - Loads one chunk at a time.
    - Splits each chunk into train/test sets.
    - Trains the model on the train set using batches (for memory efficiency within chunk).
    - Accumulates training history.
    - Returns a summary dict with loss history, etc.
    
    Args:
        model: PyTorch model (nn.Module).
        optimizer: PyTorch optimizer.
        criterion: Loss function (e.g., nn.MSELoss()).
        chunk_dir: Directory containing chunk files.
        split_ratio: Fraction for train split (e.g., 0.8).
        batch_size: Batch size for training.
        epochs_per_chunk: Number of epochs to train on each chunk.
        device: Device to train on ('cuda' or 'cpu').
        verbose: If True, print progress.
    
    Returns:
        Dict with 'train_losses' (list of lists: per-chunk losses), 'total_epochs', etc.
    """
    model.to(device)
    model.train()
    
    indices = get_chunk_indices(chunk_dir)
    train_losses = []  # List of lists: losses per epoch per chunk
    total_samples = 0
    
    for chunk_idx in indices:
        if verbose:
            print(f"Processing chunk {chunk_idx}...")
        
        # Load single chunk (assumes it fits in memory)
        noisy_chunk, label_chunk, _ = next(iter_chunks(chunk_dir, indices=[chunk_idx]))
        
        # Split into train/test (stratified if labels are categorical; here simple split)
        NP_idx = noisy_chunk.shape[1]
        shape = noisy_chunk.shape[2:]
        print('n samples in chunk:', n)
        split_fraction = 0.2      # Set the fraction of data you want to split (between 0 and 1)
        seed = 113               # Define your random seed for reproducible results

        train_NP_idx, val_NP_idx = train_test_split(NP_idx, test_size=split_fraction,
                                                    random_state=seed)

        noisy_kappa_train = noisy_chunk[:, train_NP_idx]      # shape = (Ncosmo, len(train_NP_idx), 1424, 176)
        label_train = label_chunk[:, train_NP_idx]         # shape = (Ncosmo, len(train_NP_idx), 5)
        noisy_kappa_val = noisy_chunk[:, val_NP_idx]          # shape = (Ncosmo, len(val_NP_idx), 1424, 176)
        label_val = label_chunk[:, val_NP_idx]             # shape = (Ncosmo, len(val_NP_idx), 5)

        Ntrain = label_train.shape[0]*label_train.shape[1]
        Nval = label_val.shape[0]*label_val.shape[1]
        print(f'Shape of the split training data = {noisy_kappa_train.shape}')
        print(f'Shape of the split validation data = {noisy_kappa_val.shape}')

        print(f'Shape of the split training labels = {label_train.shape}')
        print(f'Shape of the split validation labels = {label_val.shape}')
        # Reshape the data for CNN
        X_train = noisy_kappa_train.reshape(Ntrain, shape)
        X_val = noisy_kappa_val.reshape(Nval, shape)

        # Here, we ignore the nuisance parameters and only keep the 2 cosmological parameters
        y_train = label_train.reshape(Ntrain, 5)[:, :2]
        y_val = label_val.reshape(Nval, 5)[:, :2]
                        




#         tra  in_size = int(n * split_ratio)
#         # Use train_test_split for reproducibility and potential stratification
#         train_noisy, test_noisy, train_label, test_label = train_test_split(
#             noisy_chunk, label_chunk, train_size=train_size, random_state=42, shuffle=True
#         )
        
#         total_samples += train_size
        
#         # Convert to tensors for PyTorch
        train_dataset = TensorDataset(
            torch.from_numpy(X_train).float().to(device),
            torch.from_numpy(y_train).float().to(device)  # Adjust dtype if needed
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        chunk_losses = []
        for epoch in range(epochs_per_chunk):
            epoch_loss = 0.0
            num_batches = 0
            for noisy_batch, label_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(noisy_batch)
                loss = criterion(outputs, label_batch)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
                num_batches += 1
            
#             avg_epoch_loss = epoch_loss / num_batches if num_batches > 0 else 0
#             chunk_losses.append(avg_epoch_loss)
#             if verbose:
#                 print(f"  Chunk {chunk_idx}, Epoch {epoch+1}: Avg Loss = {avg_epoch_loss:.4f}")
        
#         train_losses.append(chunk_losses)
    
#     # Switch to eval mode after training
#     model.eval()
    
#     summary = {
#         'train_losses': train_losses,
#         'total_epochs': epochs_per_chunk * len(indices),
#         'total_train_samples': total_samples,
#         'num_chunks': len(indices)
#     }
    
#     if verbose:
#         print(f"Training completed. Total train samples: {total_samples}")
    
#     return summary

# # Example usage (assuming you have a model defined):
# # model = YourModel()  # e.g., nn.Sequential(nn.Linear(input_dim, hidden_dim), ...)
# # optimizer = optim.Adam(model.parameters(), lr=0.001)
# # criterion = nn.MSELoss()  # Adjust based on task
# # 
# # history = incremental_train(model, optimizer, criterion, 'dataset/chunk_kappa_noise',
# #                             batch_size=128, epochs_per_chunk=5, verbose=True)
# # 
# # print("Final train losses per chunk:", [np.mean(chunk) for chunk in history['train_losses']])

In [6]:
chunk_dir = 'dataset/chunk_kappa_noise'
indices = get_chunk_indices(chunk_dir)
train_losses = []  # List of lists: losses per epoch per chunk
total_samples = 0
    
for chunk_idx in indices:

        
        # Load single chunk (assumes it fits in memory)
    noisy_chunk, label_chunk, _ = next(iter_chunks(chunk_dir, indices=[chunk_idx]))
        
    # Split into train/test (stratified if labels are categorical; here simple split)
    n = noisy_chunk.shape[2:]
    print('n samples in chunk:', n)

n samples in chunk: (1424, 176)
n samples in chunk: (1424, 176)
n samples in chunk: (1424, 176)
n samples in chunk: (1424, 176)


KeyboardInterrupt: 

In [ ]:
# Split the data into training and validation sets

NP_idx = np.arange(Nsys)  # The indices of Nsys nuisance parameter realizations
split_fraction = 0.2      # Set the fraction of data you want to split (between 0 and 1)
seed = 5566               # Define your random seed for reproducible results

train_NP_idx, val_NP_idx = train_test_split(NP_idx, test_size=split_fraction,
                                            random_state=seed)

noisy_kappa_train = noisy_kappa[:, train_NP_idx]      # shape = (Ncosmo, len(train_NP_idx), 1424, 176)
label_train = label[:, train_NP_idx]         # shape = (Ncosmo, len(train_NP_idx), 5)
noisy_kappa_val = noisy_kappa[:, val_NP_idx]          # shape = (Ncosmo, len(val_NP_idx), 1424, 176)
label_val = label[:, val_NP_idx]             # shape = (Ncosmo, len(val_NP_idx), 5)

Ntrain = label_train.shape[0]*label_train.shape[1]
Nval = label_val.shape[0]*label_val.shape[1]